# PCA bằng SVD trên catalog sản phẩm Olist

Bảng `olist_products_dataset.csv` mô tả 32 951 sản phẩm bằng 7 cột số: khối lượng, ba kích thước,
số ảnh, độ dài tên và độ dài mô tả. Bảy cột đó có thật sự chứa bảy "chiều" thông tin không?

Bạn sẽ tự viết PCA bằng SVD, đối chiếu với `sklearn.decomposition.PCA`, kiểm tra rằng thành phần
chính là eigenvector của ma trận covariance, rồi đọc ý nghĩa các thành phần trên dữ liệu thật.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/01-math-for-ml/exercises/intermediate/03-pca-svd -q`

In [ ]:
import numpy as np
import pandas as pd

## 1. Chuẩn bị ma trận

Ba cột dài/rộng/cao và khối lượng lệch phải rất mạnh (vài gram tới 40 kg), nên lấy `log1p` trước,
rồi chuẩn hoá từng cột về trung bình 0, độ lệch chuẩn 1 (`ddof=0`). Bỏ sản phẩm thiếu bất kỳ cột
nào. Tên cột `lenght` sai chính tả là của bộ dữ liệu gốc.

In [ ]:
PRODUCT_FEATURES = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
                    "product_photos_qty", "product_name_lenght", "product_description_lenght"]


def prepare_product_matrix(products: pd.DataFrame) -> np.ndarray:
    """Ma trận n×7: log1p rồi chuẩn hoá từng cột (ddof=0), đã bỏ dòng thiếu."""
    X = np.log1p(products.dropna(subset=PRODUCT_FEATURES)[PRODUCT_FEATURES].to_numpy(dtype=float))
    return (X - X.mean(axis=0)) / X.std(axis=0)

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
products = pd.read_csv(ROOT / "data/raw/olist_products_dataset.csv")
print(len(products), "sản phẩm,", products[PRODUCT_FEATURES].isna().any(axis=1).sum(), "dòng thiếu")
X = prepare_product_matrix(products)
print(X.shape)
pd.DataFrame(np.corrcoef(X, rowvar=False), index=PRODUCT_FEATURES, columns=PRODUCT_FEATURES).round(2)

## 2. PCA bằng SVD

Với $$X_c$$ là $$X$$ đã trừ trung bình theo cột, SVD rút gọn $$X_c = U \Sigma V^\top$$:

- thành phần chính là các hàng của $$V^\top$$ (lấy `k` hàng đầu);
- variance giải thích của thành phần $$i$$ là $$\sigma_i^2 / (n - 1)$$ (khớp `sklearn`);
- tỉ lệ variance giải thích chia cho tổng **mọi** $$\sigma_i^2$$, không chỉ `k` cái đầu;
- toạ độ (scores) là $$X_c V_k$$, cũng bằng $$U_k \Sigma_k$$.

Thành phần chính chỉ xác định tới dấu. Quy ước: đổi dấu để phần tử có trị tuyệt đối lớn nhất của
mỗi thành phần là số dương (đổi cả score tương ứng).

`pca_svd(X, k)` trả dict với khoá `components` (k×d), `explained_variance` (k), `explained_variance_ratio`
(k), `scores` (n×k), `mean` (d).

In [ ]:
def pca_svd(X: np.ndarray, k: int) -> dict[str, np.ndarray]:
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    Xc = X - mean
    _, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    signs = np.sign(Vt[np.arange(len(Vt)), np.argmax(np.abs(Vt), axis=1)])
    Vt = Vt * signs[:, None]
    var = s ** 2 / (len(X) - 1)
    components = Vt[:k]
    return {
        "components": components,
        "explained_variance": var[:k],
        "explained_variance_ratio": var[:k] / var.sum(),
        "scores": Xc @ components.T,
        "mean": mean,
    }

## 3. Tái tạo và chọn số thành phần

- `reconstruct(pca)` trả $$\hat X = \text{scores} \cdot \text{components} + \text{mean}$$ (đúng kích thước X).
- `n_components_for(evr, threshold)` trả `k` nhỏ nhất để tổng tỉ lệ variance của `k` thành phần đầu
  đạt ít nhất `threshold` (`evr` là tỉ lệ của **mọi** thành phần, đã sắp giảm dần).

In [ ]:
def reconstruct(pca: dict[str, np.ndarray]) -> np.ndarray:
    return pca["scores"] @ pca["components"] + pca["mean"]


def n_components_for(evr: np.ndarray, threshold: float) -> int:
    cumulative = np.cumsum(evr)
    return int(np.searchsorted(cumulative, threshold - 1e-12) + 1)

## 4. Đọc kết quả trên dữ liệu thật

In [ ]:
# explore: so với sklearn và với eigen của covariance
from sklearn.decomposition import PCA

pca = pca_svd(X, 7)
ref = PCA().fit(X)
print("tỉ lệ variance (tự viết):", pca["explained_variance_ratio"].round(4))
print("tỉ lệ variance (sklearn):", ref.explained_variance_ratio_.round(4))
print("cộng dồn:", np.cumsum(pca["explained_variance_ratio"]).round(4))
eigval = np.linalg.eigvalsh(np.cov(X, rowvar=False))[::-1]
print("eigenvalue của covariance:", eigval.round(4))
print("σ²/(n−1) từ SVD:          ", pca["explained_variance"].round(4))

In [ ]:
# explore: loadings của hai thành phần đầu
pd.DataFrame(pca["components"][:3].T, index=PRODUCT_FEATURES, columns=["PC1", "PC2", "PC3"]).round(3)

In [ ]:
# explore: scree plot và sai số tái tạo theo k
import matplotlib.pyplot as plt

errs = [np.mean((reconstruct(pca_svd(X, k)) - X) ** 2) for k in range(1, 8)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(range(1, 8), pca["explained_variance_ratio"])
axes[0].plot(range(1, 8), np.cumsum(pca["explained_variance_ratio"]), "o-", color="black")
axes[0].set_xlabel("thành phần")
axes[0].set_title("tỉ lệ variance giải thích")
axes[1].plot(range(1, 8), errs, "o-")
axes[1].set_xlabel("k")
axes[1].set_title("MSE tái tạo")
plt.tight_layout()
plt.show()
print("k cho 90% variance:", n_components_for(pca["explained_variance_ratio"], 0.9))

In [ ]:
# explore: không chuẩn hoá thì sao? PCA trên log1p nhưng giữ nguyên thang đo của từng cột
raw = np.log1p(products.dropna(subset=PRODUCT_FEATURES)[PRODUCT_FEATURES].to_numpy(dtype=float))
unscaled = pca_svd(raw, 2)
print("tỉ lệ variance:", unscaled["explained_variance_ratio"].round(3))
print("PC1:", dict(zip(PRODUCT_FEATURES, unscaled["components"][0].round(3))))
print("độ lệch chuẩn từng cột:", dict(zip(PRODUCT_FEATURES, raw.std(axis=0).round(2))))

## 5. Nhận xét (bài mẫu)

- **PC1 (35.9% variance) là "kích thước"**: loading dương và gần bằng nhau trên khối lượng, dài,
  cao, rộng; gần 0 trên ba cột văn bản/ảnh. **PC2 (17.6%) là "độ chăm chút listing"**: số ảnh, độ dài
  tên, độ dài mô tả. Hai nhóm gần như không tương quan với nhau, nên PCA tách chúng ra rất gọn.
- **Không giảm chiều được nhiều.** Cần 5/7 thành phần để giữ 90% variance, vì ba cột văn bản/ảnh
  gần như độc lập với nhau (tương quan chỉ khoảng 0.1). PCA chỉ nén tốt khi các feature thật sự
  tương quan; ở đây chỉ khối 4 cột kích thước là nén được.
- **Chuẩn hoá thay đổi kết quả.** Không chuẩn hoá, PC1 "giải thích" 60.3% variance nhưng gần như
  chỉ là khối lượng (loading 0.91), vì log khối lượng có độ lệch chuẩn 1.35, gấp ba lần chiều dài
  (0.45). PCA chạy theo cột có variance lớn nhất, và "phần trăm variance" phụ thuộc vào đơn vị đo.
  Với feature khác đơn vị, chuẩn hoá gần như luôn là lựa chọn đúng.
- **Khi nào dùng PCA ở đây**: làm feature "kích thước" gọn cho model phí ship, hoặc trực quan hoá
  catalog. Không dùng khi cần giải thích từng feature gốc cho người làm kinh doanh, và nhớ fit PCA
  chỉ trên tập train.